# 06 — ViDeBERTa Load Forensics (is the donor actually loaded, or random?)

**The lead:** notebook 05's load report showed
`deberta.embeddings.word_embeddings._weight | UNEXPECTED` for ViDeBERTa. If that trained
embedding tensor is not being mapped onto the model's `word_embeddings.weight`, then
`AutoModel.from_pretrained('Fsoft-AIC/videberta-base')` returns a model whose word
embeddings are **random init** — and every "ViDeBERTa donor broken" number (soundness
0.014, isotropy 0.084) is an artifact of a HuggingFace load mismatch, NOT of ViDeBERTa.

**Decisive test:** compare the embedding tensor *inside the loaded model* against the
`..._weight` tensor *in the raw checkpoint file*. If they differ → the trained embeddings
never loaded. PhoBERT (which loaded cleanly in 05) is the control: its model emb must
MATCH its checkpoint, proving the comparison method itself is sound.

**If confirmed:** ViDeBERTa is vindicated, the init must be rebuilt with correctly-loaded
embeddings, and the loader (`salt3_common.load_model_safe` / notebook 01 cell 7) needs a
`_weight`→`weight` remap. We do not act until the diff is on screen.


In [1]:
%%capture
!pip install -U transformers safetensors huggingface_hub fasttext-wheel datasets


In [2]:
import sys
from pathlib import Path
import numpy as np, torch
import torch.nn.functional as F
import transformers
from transformers import AutoModel, AutoModelForMaskedLM, AutoTokenizer

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)

PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3')
sys.path.insert(0, str(PROJECT_ROOT / 'code')); sys.path.insert(0, '/content')
import importlib, salt3_diagnostics as dx
importlib.reload(dx)

print('transformers version:', transformers.__version__)
print('torch version       :', torch.__version__)
VIDE_ID, PHO_ID = 'Fsoft-AIC/videberta-base', 'vinai/phobert-base-v2'
FT_VI_BIN = str(PROJECT_ROOT / 'init' / 'videberta_salt_init_v5_globalmap_freqbias' / 'cc.vi.300.bin')


Mounted at /content/drive
transformers version: 5.10.2
torch version       : 2.11.0+cu128


## A. What the loaded model holds
The embedding tensor `AutoModel` actually gives us (what the whole pipeline has been using).
A std near ~0.02 with no rogue/common direction is the fingerprint of fresh random init.


In [3]:
def model_emb(model):
    return model.get_input_embeddings().weight.detach().float().cpu()

vide_model = AutoModel.from_pretrained(VIDE_ID, trust_remote_code=True)
vide_tok = AutoTokenizer.from_pretrained(VIDE_ID)
m_vide = model_emb(vide_model)
print(f'ViDeBERTa model emb: shape {tuple(m_vide.shape)}  std {m_vide.std():.5f}  '
      f'mean-row-norm {m_vide.norm(dim=1).mean():.4f}')
print(f'  row[100][:6] = {m_vide[100][:6].tolist()}')

pho_model = AutoModel.from_pretrained(PHO_ID)
pho_tok = AutoTokenizer.from_pretrained(PHO_ID)
m_pho = model_emb(pho_model)
print(f'PhoBERT  model emb: shape {tuple(m_pho.shape)}  std {m_pho.std():.5f}  '
      f'mean-row-norm {m_pho.norm(dim=1).mean():.4f}')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/610 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/567M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: Fsoft-AIC/videberta-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

PhoBERT  model emb: shape (64001, 768)  std 0.06150  mean-row-norm 1.6980


## B. What the raw checkpoint file holds
Pull the actual state dict and find the `word_embeddings` tensor (note the `_weight` key).
Also list every checkpoint key the model did NOT consume (unexpected) and every model key
left at init (missing) — that is the load-bug surface.


In [4]:
from huggingface_hub import hf_hub_download

def load_state_dict(repo):
    for fn in ('model.safetensors', 'pytorch_model.bin'):
        try:
            p = hf_hub_download(repo_id=repo, filename=fn)
        except Exception:
            continue
        if fn.endswith('.safetensors'):
            from safetensors.torch import load_file
            return load_file(p), fn
        return torch.load(p, map_location='cpu', weights_only=True), fn
    raise FileNotFoundError(repo)

vide_sd, vide_file = load_state_dict(VIDE_ID)
print(f'ViDeBERTa checkpoint file: {vide_file},  {len(vide_sd)} tensors')
emb_keys = [k for k in vide_sd if 'word_embeddings' in k]
print('  word_embeddings keys in checkpoint:', emb_keys)
ck_key = next((k for k in emb_keys if k.endswith('_weight') or k.endswith('weight')), None)
ck_vide = vide_sd[ck_key].float()
print(f'  checkpoint {ck_key}: shape {tuple(ck_vide.shape)}  std {ck_vide.std():.5f}  '
      f'mean-row-norm {ck_vide.norm(dim=1).mean():.4f}')
print(f'  row[100][:6] = {ck_vide[100][:6].tolist()}')

# key-level load audit (strip the 'deberta.'/'roberta.' base-model prefix the way HF does)
model_keys = set(vide_model.state_dict().keys())
def strip_prefix(k):
    for p in ('deberta.', 'roberta.', 'model.'):
        if k.startswith(p):
            return k[len(p):]
    return k
unexpected = [k for k in vide_sd if strip_prefix(k) not in model_keys]
missing = [k for k in model_keys if k not in {strip_prefix(x) for x in vide_sd}]
print(f'  checkpoint keys NOT consumed by model (unexpected): {len(unexpected)} e.g. {unexpected[:4]}')
print(f'  model keys left at INIT (missing from ckpt): {len(missing)} e.g. {missing[:4]}')


ViDeBERTa checkpoint file: pytorch_model.bin,  207 tensors
  word_embeddings keys in checkpoint: ['deberta.embeddings.word_embeddings._weight', 'deberta.embeddings.word_embeddings.weight']
  checkpoint deberta.embeddings.word_embeddings._weight: shape (128000, 768)  std 0.10534  mean-row-norm 2.8477
  row[100][:6] = [-0.2127685546875, 0.00705718994140625, -0.0309295654296875, -0.0418701171875, -0.0780029296875, 0.1951904296875]
  checkpoint keys NOT consumed by model (unexpected): 9 e.g. ['deberta.embeddings.word_embeddings._weight', 'deberta.embeddings.position_embeddings._weight', 'deberta.embeddings.position_embeddings.weight', 'mask_predictions.dense.weight']
  model keys left at INIT (missing from ckpt): 0 e.g. []


## C. Decisive comparison — did the trained embeddings reach the model?
PhoBERT is the control: its model emb MUST equal its checkpoint emb (clean load). If
ViDeBERTa's model emb does NOT equal its checkpoint emb, the trained embeddings never
loaded and the donor was random all along.


In [5]:
def compare(name, model_e, ck_e):
    n = min(model_e.shape[0], ck_e.shape[0])
    a, b = model_e[:n], ck_e[:n]
    max_diff = (a - b).abs().max().item()
    row_cos = float(F.cosine_similarity(a[100:200], b[100:200]).mean())
    same = max_diff < 1e-4
    print(f'{name:10s}: max|model-ckpt| {max_diff:.4e}  mean row cos {row_cos:+.3f}  '
          f'-> {"MATCH (loaded)" if same else "DIFFERENT (NOT loaded -> random in model)"}')
    return same

# control
pho_sd, pho_file = load_state_dict(PHO_ID)
pk = next(k for k in pho_sd if 'word_embeddings' in k and (k.endswith('weight') or k.endswith('_weight')))
print(f'PhoBERT checkpoint emb key: {pk}')
pho_loaded = compare('PhoBERT', m_pho, pho_sd[pk].float())
vide_loaded = compare('ViDeBERTa', m_vide, ck_vide)

assert pho_loaded, 'CONTROL FAILED: PhoBERT should match — comparison method is wrong, stop and rethink.'
print('\ncontrol OK (PhoBERT matches) -> the ViDeBERTa verdict above is trustworthy.')


PhoBERT checkpoint emb key: roberta.embeddings.word_embeddings.weight
PhoBERT   : max|model-ckpt| 0.0000e+00  mean row cos +1.000  -> MATCH (loaded)
ViDeBERTa : max|model-ckpt| 2.0654e+00  mean row cos +0.780  -> DIFFERENT (NOT loaded -> random in model)

control OK (PhoBERT matches) -> the ViDeBERTa verdict above is trustworthy.


## D. Semantic neighbor proof (human-readable)
Nearest neighbors of a few Vietnamese words in the MODEL emb vs the CHECKPOINT emb. If the
checkpoint gives meaningful neighbors and the model gives noise, the bug is visible by eye.


In [6]:
import fasttext  # noqa
vide_vocab = vide_tok.get_vocab()
inv = {i: t for t, i in vide_vocab.items()}

def neighbors(emb, token, topn=8):
    if token not in vide_vocab:
        return ['<oov>']
    E = F.normalize(emb.float(), dim=1)
    q = E[vide_vocab[token]]
    idx = (E @ q).topk(topn + 1).indices[1:].tolist()
    return [inv.get(i, '?') for i in idx]

for w in ['▁Việt', '▁kinh', '▁bệnh', '▁trường', '▁ăn']:
    if w in vide_vocab:
        print(f'{w!r}')
        print(f'   model  : {neighbors(m_vide, w)}')
        print(f'   ckpt   : {neighbors(ck_vide, w)}')


'▁Việt'
   model  : ['▁con', '▁trông', '▁đăng', '▁xác', 'blanc', '+', '▁hãng', '▁xác_định']
   ckpt   : ['blanc', '▁thân', '▁đăng', '▁con', 'Quốc_Tuấn', '▁chuyên', '▁khả_năng', '▁1.4']
'▁kinh'
   model  : ['▁thương_hiệu', '▁LH', '▁trai', '▁môi', 'tín_dụng', '_phận', '▁hành_chính', '▁TỰ_']
   ckpt   : ['▁TỰ_', '▁thương_hiệu', '▁LH', 'tín_dụng', '▁trai', '▁nước_tiểu', '▁môi', '1845']
'▁bệnh'
   model  : ['▁đồng', '▁khá', '▁độ', '▁doanh_nghiệp', '▁tuổi', '▁Bài', '▁Phải', '▁dữ']
   ckpt   : ['▁đồng', '▁độ', '▁tuổi', '▁M', '▁doanh_nghiệp', 'މ', '▁có', '▁trong']
'▁trường'
   model  : ['▁đăng_nhập', '▁Xóa', '람', '205', '▁S', '\uf69b', 'Y', 'ِ']
   ckpt   : ['▁Xóa', '▁đăng_nhập', '\uf69b', '205', '▁S', '람', '_ấm', 'ِ']
'▁ăn'
   model  : ['sinh_lực', '▁buồn', '_Pedro', '▁soạn_thảo', 'Director', '▁B', '▁xe', '▁học']
   ckpt   : ['▁buồn', '▁băng_', 'Ung_thư', '▁LÃ', '▁NQ', 'Director', 'taku', 'trễ']


## E. Apply the fix and re-measure soundness
Inject the checkpoint `_weight` into the embedding matrix and re-run donor soundness.
0.014 (broken) vs whatever the correctly-loaded ViDeBERTa scores = the entire story.


In [7]:
ft_vi = fasttext.load_model(FT_VI_BIN)

print('BEFORE (model emb as the pipeline loaded it):')
sound_before = dx.donor_space_soundness(m_vide, vide_vocab, ft_vi, n=1000, k=10)
print('\nAFTER (checkpoint _weight injected):')
sound_after = dx.donor_space_soundness(ck_vide, vide_vocab, ft_vi, n=1000, k=10)


BEFORE (model emb as the pipeline loaded it):


── Donor-space soundness (n=1000, k=10, chance≈0.010) ──
  kNN overlap vs FastText: raw 0.014 | syllable-mean 0.011
  mean-centered          : raw 0.014 | syllable-mean 0.013
  verdict (best variant) : DONOR BROKEN — ~chance vs FastText; swap donor   (gates: >0.2 usable, <0.05 broken)

AFTER (checkpoint _weight injected):
── Donor-space soundness (n=1000, k=10, chance≈0.010) ──
  kNN overlap vs FastText: raw 0.012 | syllable-mean 0.011
  mean-centered          : raw 0.013 | syllable-mean 0.013
  verdict (best variant) : DONOR BROKEN — ~chance vs FastText; swap donor   (gates: >0.2 usable, <0.05 broken)


## F. Verdict

In [8]:
print('=' * 68); print('VIDEBERTA LOAD FORENSICS — VERDICT'); print('=' * 68)
print(f'transformers {transformers.__version__}')
print(f'PhoBERT control match (method valid): {pho_loaded}')
print(f'ViDeBERTa model emb == checkpoint emb: {vide_loaded}')
print(f'model emb std {m_vide.std():.4f}  vs  checkpoint emb std {ck_vide.std():.4f}')
b_before = max(sound_before["overlap_raw"], sound_before["overlap_raw_centered"])
b_after = max(sound_after["overlap_raw"], sound_after["overlap_raw_centered"])
print(f'soundness  before {b_before:.3f}  ->  after fix {b_after:.3f}')
print('-' * 68)
if not vide_loaded and b_after > 0.2:
    print('CONFIRMED: load bug. ViDeBERTa embeddings were RANDOM in the model; the real')
    print('trained embeddings carry VI semantics. Action: fix the loader (_weight->weight)')
    print('in notebook 01 cell 7 / salt3_common, then REBUILD the init. Donor stays ViDeBERTa.')
elif not vide_loaded and b_after <= 0.2:
    print('Embeddings were not loaded, but even the correct ones score low — investigate')
    print('the checkpoint format further before any donor decision.')
else:
    print('Embeddings WERE loaded correctly; the low soundness is real. Re-open the donor')
    print('question with this confirmed.')
print('=' * 68)


VIDEBERTA LOAD FORENSICS — VERDICT
transformers 5.10.2
PhoBERT control match (method valid): True
ViDeBERTa model emb == checkpoint emb: False
model emb std 0.1276  vs  checkpoint emb std 0.1053
soundness  before 0.014  ->  after fix 0.013
--------------------------------------------------------------------
Embeddings were not loaded, but even the correct ones score low — investigate
the checkpoint format further before any donor decision.
